In [1]:
import cv2
import numpy as np
import pandas as pd
import os

from skimage.feature import graycomatrix, graycoprops
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
folder_path = '/content/drive/MyDrive/ekstarsi tgs omvis' # ganti sesuai folder kamu

image_files = os.listdir(folder_path)
print("Jumlah gambar:", len(image_files))

Jumlah gambar: 10


In [5]:
def extract_features(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Resize biar konsisten
    gray = cv2.resize(gray, (256, 256))

    # --- GLCM ---
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]

    # --- Histogram ---
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist.flatten()

    mean = np.mean(hist)
    std = np.std(hist)

    return contrast, energy, homogeneity, correlation, mean, std

In [6]:
data = []

for file in image_files:
    path = os.path.join(folder_path, file)

    try:
        contrast, energy, homogeneity, correlation, mean, std = extract_features(path)

        data.append([file, contrast, energy, homogeneity, correlation, mean, std])

    except:
        print("Error di file:", file)

In [7]:
columns = ['Nama Gambar', 'Contrast', 'Energy', 'Homogeneity', 'Correlation', 'Mean', 'Std']

df = pd.DataFrame(data, columns=columns)

df.head()

,Nama Gambar,Contrast,Energy,Homogeneity,Correlation,Mean,Std
0,IMG_7694.JPG,132.971982,0.021030,0.265107,0.982882,256.0,200.901199
1,IMG_7740.JPG,196.129228,0.021361,0.180490,0.976135,256.0,191.642487
2,IMG_7698.JPG,79.499602,0.019882,0.239729,0.983315,256.0,197.889404
3,IMG_7612.JPG,72.383226,0.218407,0.412576,0.995422,256.0,951.231750
4,IMG_7758.JPG,125.202819,0.018224,0.166840,0.971239,256.0,264.675262


In [8]:
output_path = '/content/drive/MyDrive/hasil_ekstraksi.csv'
df.to_csv(output_path, index=False)

print("File berhasil disimpan di:", output_path)

File berhasil disimpan di: /content/drive/MyDrive/hasil_ekstraksi.csv
